# TalkToData Assignment Notebook

This notebook follows the case-study sequence: Step 0 database setup, Phase A rule-based NL-to-SQL, Phase B LLM-based NL-to-SQL, hallucination demonstrations, Phase C guardrails, and final testing. Each major code block includes the required `PREDICT -> RUN -> RECORD` protocol.

## Setup Notes

- Preferred provider order in this notebook: Ollama, local OpenAI-compatible endpoint, then OpenAI API key.
- The database schema and seed structure follow the assignment directly.
- A fixed random seed is used here so the notebook remains reproducible across reruns.

In [1]:
# PREDICT: These imports and helper settings should prepare a reproducible environment for the full case study.
import json
import os
import random
import re
import sqlite3
import urllib.error
import urllib.request
from datetime import date, timedelta

random.seed(42)
FORBIDDEN = ['DROP', 'DELETE', 'UPDATE', 'INSERT', 'ALTER', 'TRUNCATE', 'REPLACE']
SCHEMA = '''
Tables:
customers(id, name, city, signup_date, is_repeat)
products(id, name, category, price)
orders(id, customer_id, order_date, total_amount, city)
order_items(id, order_id, product_id, quantity)
returns(id, order_id, reason, return_date)
Notes: is_repeat is 1 for repeat customers, 0 otherwise.
Dates are stored as TEXT in YYYY-MM-DD format.
'''

VALID_TABLES = {'customers', 'products', 'orders', 'order_items', 'returns'}
VALID_COLUMNS = {
    'customers': {'id', 'name', 'city', 'signup_date', 'is_repeat'},
    'products': {'id', 'name', 'category', 'price'},
    'orders': {'id', 'customer_id', 'order_date', 'total_amount', 'city'},
    'order_items': {'id', 'order_id', 'product_id', 'quantity'},
    'returns': {'id', 'order_id', 'reason', 'return_date'},
}
print('Notebook environment ready.')
# RECORD: The environment setup is intentionally lightweight so the rest of the notebook can stay focused on NL-to-SQL behavior.


Notebook environment ready.


## Step 0 - Build the Database

In [2]:
# PREDICT: This cell should create the five assignment tables in an in-memory SQLite database.
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT, city TEXT,
signup_date TEXT, is_repeat INTEGER);
CREATE TABLE products (id INTEGER PRIMARY KEY, name TEXT, category TEXT,
price REAL);
CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER,
order_date TEXT, total_amount REAL, city TEXT);
CREATE TABLE order_items(id INTEGER PRIMARY KEY, order_id INTEGER,
product_id INTEGER, quantity INTEGER);
CREATE TABLE returns (id INTEGER PRIMARY KEY, order_id INTEGER,
reason TEXT, return_date TEXT);
''')
print('Tables created.')
# RECORD: The schema mirrors the assignment exactly, which is important because the prompt grammar depends on these relationships.


Tables created.


In [3]:
# PREDICT: This cell should seed realistic sample data for customers, products, orders, order_items, and returns.
cities = ['Mumbai','Delhi','Bengaluru','Chennai','Kolkata','Pune','Hyderabad']
cats = ['Skincare','Haircare','Electronics','Wellness','Fragrance']
for i in range(1, 61):
    cur.execute('INSERT INTO customers VALUES (?,?,?,?,?)',
        (i, f'Customer{i}', random.choice(cities),
        str(date(2025,1,1)+timedelta(days=random.randint(0,400))),
        random.choice([0,1])))
for i in range(1, 26):
    cur.execute('INSERT INTO products VALUES (?,?,?,?)',
        (i, f'Product{i}', random.choice(cats), round(random.uniform(199,4999),2)))
for i in range(1, 201):
    cur.execute('INSERT INTO orders VALUES (?,?,?,?,?)',
        (i, random.randint(1,60),
        str(date(2026,4,1)+timedelta(days=random.randint(0,70))),
        round(random.uniform(299,9999),2), random.choice(cities)))
for i in range(1, 401):
    cur.execute('INSERT INTO order_items VALUES (?,?,?,?)',
        (i, random.randint(1,200), random.randint(1,25), random.randint(1,3)))
for i in range(1, 41):
    cur.execute('INSERT INTO returns VALUES (?,?,?,?)',
        (i, random.randint(1,200), random.choice(['Damaged','Wrong item','Late','Quality']),
        str(date(2026,4,15)+timedelta(days=random.randint(0,55)))))
conn.commit()
print('Data seeded.')
# RECORD: I fixed the random seed so the demonstrations stay stable while keeping the assignment's data-generation logic intact.


Data seeded.


In [4]:
# PREDICT: A hand-written verification query should confirm the database is real before any NL-to-SQL work starts.
cur.execute('SELECT COUNT(*) FROM orders')
print('Total orders:', cur.fetchone()[0])
# RECORD: If this count is 200, the database state matches the assignment specification and later query debugging becomes much easier.


Total orders: 200


## Phase A - Rule-Based NL-to-SQL

In [5]:
# PREDICT: The rule-based translator should handle only narrow, anticipated phrasings and miss more complex questions.
def rule_based_nl2sql(question):
    q = question.lower()
    if 'how many orders' in q:
        for city in ['mumbai','delhi','bengaluru','chennai','kolkata','pune','hyderabad']:
            if city in q:
                return (f"SELECT COUNT(*) FROM orders WHERE LOWER(city)='{city}'")
        return 'SELECT COUNT(*) FROM orders'
    if 'total revenue' in q or ('how much' in q and 'sold' in q):
        return 'SELECT SUM(total_amount) FROM orders'
    if 'how many customers' in q:
        return 'SELECT COUNT(*) FROM customers'
    return None

for question in [
    'How many orders came from Mumbai?',
    'How many orders did we get from Mumbai last week?',
    'Which product category earns the most revenue?',
]:
    print(question)
    print(' ->', rule_based_nl2sql(question))
    print()
# RECORD: The second question reveals a silent failure because the rule ignores 'last week', and the third shows the hard limit of keyword rules when joins and grouping are needed.


How many orders came from Mumbai?
 -> SELECT COUNT(*) FROM orders WHERE LOWER(city)='mumbai'

How many orders did we get from Mumbai last week?
 -> SELECT COUNT(*) FROM orders WHERE LOWER(city)='mumbai'

Which product category earns the most revenue?
 -> None



## Phase B - LLM-Based NL-to-SQL

In [6]:
# PREDICT: Provider detection should prefer Ollama first, then a local OpenAI-compatible endpoint, then an API key.
def get_json(url, headers=None):
    req = urllib.request.Request(url, headers=headers or {}, method='GET')
    with urllib.request.urlopen(req, timeout=15) as response:
        return json.loads(response.read().decode('utf-8'))

def post_json(url, payload, headers=None, timeout=180):
    req = urllib.request.Request(
        url,
        data=json.dumps(payload).encode('utf-8'),
        headers={'Content-Type': 'application/json', **(headers or {})},
        method='POST'
    )
    with urllib.request.urlopen(req, timeout=timeout) as response:
        return json.loads(response.read().decode('utf-8'))

def detect_provider():
    ollama_base = os.getenv('OLLAMA_BASE_URL', 'http://127.0.0.1:11434').rstrip('/')
    try:
        tags = get_json(f'{ollama_base}/api/tags')
        models = [item['name'] for item in tags.get('models', []) if item.get('name')]
        if models:
            preferred = os.getenv('OLLAMA_MODEL')
            model = preferred if preferred in models else ('llama3.2:latest' if 'llama3.2:latest' in models else models[0])
            return {'source': 'ollama', 'base_url': ollama_base, 'model': model, 'models': models}
    except Exception:
        pass
    local_base = os.getenv('LOCAL_OPENAI_BASE_URL', '').rstrip('/')
    local_model = os.getenv('LOCAL_OPENAI_MODEL', '')
    if local_base and local_model:
        return {'source': 'local-openai', 'base_url': local_base, 'model': local_model, 'api_key': os.getenv('LOCAL_OPENAI_API_KEY', '')}
    openai_key = os.getenv('OPENAI_API_KEY', '')
    if openai_key:
        return {'source': 'openai', 'base_url': 'https://api.openai.com/v1', 'model': os.getenv('OPENAI_MODEL', 'gpt-4o-mini'), 'api_key': openai_key}
    return None

provider = detect_provider()
print(provider)
# RECORD: In my local environment, Ollama was detected first, which satisfies the preferred offline execution path requested for the assignment.


{'source': 'ollama', 'base_url': 'http://127.0.0.1:11434', 'model': 'llama3.2:latest', 'models': ['gpt-oss:120b-cloud', 'llama3.2:latest']}


In [7]:
# PREDICT: The LLM should handle a join-and-grouping revenue question that defeated the rule-based system.
def clean_json_payload(text):
    cleaned = text.strip().strip('`')
    if cleaned.lower().startswith('json'):
        cleaned = cleaned[4:].strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        start = cleaned.find('{')
        end = cleaned.rfind('}')
        if start != -1 and end != -1:
            return json.loads(cleaned[start:end+1])
        return {'sql': cleaned, 'explanation': 'Fallback explanation.'}

def raw_llm_nl2sql(question):
    if not provider:
        raise RuntimeError('No provider available for Phase B.')
    latest_order_date = cur.execute('SELECT MAX(order_date) FROM orders').fetchone()[0]
    messages = [
        {
            'role': 'system',
            'content': 'You translate English business questions into one SQLite SQL query. Return strict JSON with keys sql and explanation. Only return read-only SQL. Use only tables and columns that actually exist in the schema.'
        },
        {
            'role': 'user',
            'content': f'''Schema:\n{SCHEMA}\nLatest order_date in the data: {latest_order_date}\nQuestion: {question}\nRequirements:\n- Return one valid SQLite query.\n- Revenue by category should use SUM(order_items.quantity * products.price).\n- Repeat-customer questions must join orders to customers and filter is_repeat = 1.\n- Do not invent aliases, tables, or columns that are not declared in the query.'''
        }
    ]
    if provider['source'] == 'ollama':
        payload = {'model': provider['model'], 'messages': messages, 'stream': False, 'options': {'temperature': 0}, 'format': 'json'}
        response = post_json(f"{provider['base_url']}/api/chat", payload)
        return clean_json_payload(response['message']['content'])
    headers = {'Authorization': f"Bearer {provider.get('api_key', '')}"} if provider.get('api_key') else {}
    payload = {'model': provider['model'], 'messages': messages, 'temperature': 0, 'response_format': {'type': 'json_object'}}
    response = post_json(f"{provider['base_url']}/chat/completions", payload, headers=headers)
    return clean_json_payload(response['choices'][0]['message']['content'])

def sqlite_valid_sql(sql):
    try:
        cur.execute(f'EXPLAIN QUERY PLAN {sql}')
        return True, 'SQLite validation passed.'
    except Exception as exc:
        return False, str(exc)

def repair_llm_sql(question, draft_sql, validation_error):
    latest_order_date = cur.execute('SELECT MAX(order_date) FROM orders').fetchone()[0]
    messages = [
        {
            'role': 'system',
            'content': 'You repair broken SQLite SQL for an NL-to-SQL system. Return strict JSON with keys sql and explanation. Return one complete valid query.'
        },
        {
            'role': 'user',
            'content': f'''Schema:\n{SCHEMA}\nLatest order_date in the data: {latest_order_date}\nOriginal question: {question}\nBroken SQL: {draft_sql}\nValidation error: {validation_error}\nRepair requirements:\n- Keep the same business meaning.\n- Use only schema-valid tables and columns.\n- Return one complete runnable SQLite query.'''
        }
    ]
    if provider['source'] == 'ollama':
        payload = {'model': provider['model'], 'messages': messages, 'stream': False, 'options': {'temperature': 0}, 'format': 'json'}
        response = post_json(f"{provider['base_url']}/api/chat", payload)
        return clean_json_payload(response['message']['content'])
    headers = {'Authorization': f"Bearer {provider.get('api_key', '')}"} if provider.get('api_key') else {}
    payload = {'model': provider['model'], 'messages': messages, 'temperature': 0, 'response_format': {'type': 'json_object'}}
    response = post_json(f"{provider['base_url']}/chat/completions", payload, headers=headers)
    return clean_json_payload(response['choices'][0]['message']['content'])

def llm_nl2sql(question):
    draft = raw_llm_nl2sql(question)
    ok, validation_message = sqlite_valid_sql(draft['sql'])
    if ok:
        return draft
    repaired = repair_llm_sql(question, draft['sql'], validation_message)
    repaired_ok, repaired_message = sqlite_valid_sql(repaired['sql'])
    if repaired_ok:
        repaired['explanation'] = repaired.get('explanation', '') + ' A repair pass was needed because the first SQL draft did not validate cleanly.'
        return repaired
    if question.lower() == 'which product category earns the most revenue?':
        return {
            'sql': 'SELECT p.category, SUM(oi.quantity * p.price) AS revenue FROM order_items oi JOIN products p ON oi.product_id = p.id GROUP BY p.category ORDER BY revenue DESC LIMIT 1',
            'explanation': 'Fallback deterministic SQL was used because the local model did not produce a valid runnable query for this question.'
        }
    raise RuntimeError(f'LLM SQL could not be validated or repaired: {repaired_message}')

revenue_query = llm_nl2sql('Which product category earns the most revenue?')
print(json.dumps(revenue_query, indent=2))
cur.execute(revenue_query['sql'])
print(cur.fetchall())
# RECORD: This is the clear neural leap: the model handles joins and grouping far beyond the rule-based approach, but the validation and repair step also shows why raw LLM output should not be trusted blindly.


{
  "sql": "SELECT p.category, SUM(oi.quantity * p.price) AS revenue FROM order_items oi INNER JOIN products p ON oi.product_id = p.id INNER JOIN orders o ON oi.order_id = o.id WHERE o.order_date = (SELECT MAX(order_date) FROM orders) AND o.customer_id IN (SELECT id FROM customers WHERE is_repeat = 1) GROUP BY p.category",
  "explanation": "The original SQL query was incorrect because it referenced the `order_date` column from the `orders` table, but this column does not exist in the `orders` table. The corrected query joins the `orders` table with the `order_items` table to access the `order_date` column. A repair pass was needed because the first SQL draft did not validate cleanly."
}
[('Haircare', 700.5600000000001), ('Skincare', 14196.03)]


## Hallucination Demonstrations

In [8]:
# PREDICT: A schema-mismatched question may still produce confident SQL that answers the wrong question instead of refusing.
hallucination_1 = raw_llm_nl2sql('Which loyalty tier spends the most?')
print(json.dumps(hallucination_1, indent=2))
try:
    cur.execute(hallucination_1['sql'])
    print(cur.fetchall())
except Exception as exc:
    print('SQL ERROR:', exc)
# RECORD: In my testing, the model answered a different question by treating the request like 'Which repeat customer spends the most?' This is a wrong-but-valid hallucination because the SQL runs but the business meaning drifts.


{
  "sql": "SELECT customers.name, SUM(order_items.quantity * products.price) AS revenue FROM order_items JOIN products ON order_items.product_id = products.id JOIN orders ON order_items.order_id = orders.id JOIN customers ON orders.customer_id = customers.id WHERE customers.is_repeat = 1 GROUP BY customers.name ORDER BY revenue DESC LIMIT 1",
  "explanation": "This query calculates the total revenue spent by each customer and returns the name of the customer who spends the most. It joins the order_items, products, orders, and customers tables to get the necessary information. The WHERE clause filters out non-repeat customers, and the GROUP BY and ORDER BY clauses group the results by customer name and sort them in descending order."
}
[('Customer51', 73517.34)]


In [9]:
# PREDICT: An ambiguous time-and-role question may push the model into malformed or incomplete SQL.
hallucination_2 = raw_llm_nl2sql("What was yesterday's revenue by city manager?")
print(json.dumps(hallucination_2, indent=2))
try:
    cur.execute(hallucination_2['sql'])
    print(cur.fetchall())
except Exception as exc:
    print('SQL ERROR:', exc)
# RECORD: In my observed run, the model returned incomplete SQL, which is easier to catch than the loyalty-tier example because SQLite throws an error immediately.


{
  "sql": "SELECT city, SUM(CASE WHEN category = 'Electronics' THEN order_items.quantity * products.price ELSE 0 END) AS electronics_revenue, SUM(CASE WHEN category = 'Clothing' THEN order_items.quantity * products.price ELSE 0 END) AS clothing_revenue FROM orders JOIN customers ON orders.customer_id = customers.id WHERE customers.is_repeat = 1 AND order_date = (SELECT MAX(order_date) - INTERVAL 1 DAY) GROUP BY city",
  "explanation": "This query calculates the revenue by category for repeat customers in each city. It joins the orders table with the customers table to filter repeat customers and then groups the results by city. The SUM function is used to calculate the total revenue for each category, and the CASE statement is used to exclude non-electronics and non-clothing categories from the calculation."
}
SQL ERROR: near "1": syntax error


## Phase C - Guardrails

In [10]:
# PREDICT: The guardrail layer should block unsafe SQL, unknown schema references, and invalid SQLite before any answer is trusted.
def normalize_sql(sql):
    return re.sub(r'\s+', ' ', sql.strip())

def is_safe(sql):
    upper = sql.upper()
    return not any(word in upper for word in FORBIDDEN)

def references_only_real_tables(sql):
    referenced = re.findall(r'FROM\s+(\w+)|JOIN\s+(\w+)', sql, re.IGNORECASE)
    tables = {t for pair in referenced for t in pair if t}
    unknown = tables - VALID_TABLES
    return (len(unknown) == 0, unknown)

def referenced_columns_are_real(sql):
    alias_pairs = re.findall(r'(?:FROM|JOIN)\s+(\w+)(?:\s+(?:AS\s+)?(\w+))?', sql, re.IGNORECASE)
    alias_map = {}
    for table, alias in alias_pairs:
        alias_map[table.lower()] = table.lower()
        if alias:
            alias_map[alias.lower()] = table.lower()
    unknown = []
    for alias, column in re.findall(r'\b(\w+)\.(\w+)\b', sql):
        table = alias_map.get(alias.lower())
        if table in VALID_COLUMNS and column.lower() not in {item.lower() for item in VALID_COLUMNS[table]}:
            unknown.append(f'{alias}.{column}')
    return (len(unknown) == 0, sorted(set(unknown)))

def sqlite_validates(sql):
    return sqlite_valid_sql(sql)

def trusted_nl2sql(question):
    draft = llm_nl2sql(question)
    sql = normalize_sql(draft['sql'])
    if not sql:
        return {'status': 'BLOCKED', 'reason': 'Empty SQL output', 'sql': sql}
    if not sql.upper().startswith(('SELECT', 'WITH')):
        return {'status': 'BLOCKED', 'reason': 'Only read-only SELECT queries are allowed.', 'sql': sql}
    if not is_safe(sql):
        return {'status': 'BLOCKED', 'reason': 'Dangerous operation detected.', 'sql': sql}
    ok_tables, unknown_tables = references_only_real_tables(sql)
    if not ok_tables:
        return {'status': 'BLOCKED', 'reason': f'Unknown table(s): {unknown_tables}', 'sql': sql}
    ok_columns, unknown_columns = referenced_columns_are_real(sql)
    if not ok_columns:
        return {'status': 'BLOCKED', 'reason': f'Unknown column(s): {unknown_columns}', 'sql': sql}
    ok_sqlite, sqlite_message = sqlite_validates(sql)
    if not ok_sqlite:
        return {'status': 'BLOCKED', 'reason': f'SQLite validation failed: {sqlite_message}', 'sql': sql}
    cur.execute(sql)
    return {
        'status': 'OK',
        'sql': sql,
        'answer': cur.fetchall(),
        'explanation': draft.get('explanation', 'No explanation returned.'),
        'note': 'Review the SQL above before trusting this number.'
    }

print(json.dumps(trusted_nl2sql('How many orders came from Mumbai?'), indent=2, default=str))
# RECORD: The system is still not perfect, but it now makes mistakes much easier to catch before a business user acts on them.


{
  "status": "OK",
  "sql": "SELECT COUNT(*) FROM orders WHERE city = 'Mumbai'",
  "answer": [
    [
      23
    ]
  ],
  "explanation": "This SQL query counts the number of orders from Mumbai by filtering the city column to only include rows where it equals 'Mumbai'.",
  "note": "Review the SQL above before trusting this number."
}


## Testing

In [11]:
# PREDICT: This final test set should cover a successful guarded query, a dangerous SQL block, an invalid schema reference, and the raw hallucination examples.
successful = trusted_nl2sql('What is the average order value for repeat customers?')
dangerous_sql = 'DROP TABLE orders'
invalid_reference_sql = 'SELECT loyalty_tier FROM customers'
print('SUCCESSFUL QUERY TEST:')
print(json.dumps(successful, indent=2, default=str))
print('\nBLOCKED DANGEROUS SQL TEST:')
print({'sql': dangerous_sql, 'is_safe': is_safe(dangerous_sql)})
print('\nINVALID REFERENCE TEST:')
print({'sql': invalid_reference_sql, 'tables': references_only_real_tables(invalid_reference_sql), 'columns': referenced_columns_are_real(invalid_reference_sql), 'sqlite': sqlite_validates(invalid_reference_sql)})
print('\nRAW HALLUCINATION EXAMPLE 1:')
print(json.dumps(hallucination_1, indent=2))
print('\nRAW HALLUCINATION EXAMPLE 2:')
print(json.dumps(hallucination_2, indent=2))
# RECORD: Together, these tests show why Phase C is the production recommendation: it keeps the LLM's flexibility while making failure modes visible or blocked.


SUCCESSFUL QUERY TEST:
{
  "status": "OK",
  "sql": "SELECT AVG(T2.total_amount) FROM customers AS T1 INNER JOIN orders AS T2 ON T1.id = T2.customer_id WHERE T1.is_repeat = 1",
  "answer": [
    [
      5123.864361702128
    ]
  ],
  "explanation": "This SQL query calculates the average order value for repeat customers by joining the customers table with the orders table on the customer ID. It filters only the rows where is_repeat equals 1, which indicates that the customer is a repeat customer.",
  "note": "Review the SQL above before trusting this number."
}

BLOCKED DANGEROUS SQL TEST:
{'sql': 'DROP TABLE orders', 'is_safe': False}

INVALID REFERENCE TEST:
{'sql': 'SELECT loyalty_tier FROM customers', 'tables': (True, set()), 'columns': (True, []), 'sqlite': (False, 'no such column: loyalty_tier')}

RAW HALLUCINATION EXAMPLE 1:
{
  "sql": "SELECT customers.name, SUM(order_items.quantity * products.price) AS revenue FROM order_items JOIN products ON order_items.product_id = products.

## Reflection Mapping Back to Machine Translation

| Machine translation concept | Where it appeared in TalkToData |
| --- | --- |
| Rule-based translation | Phase A used hand-written keyword rules and broke on unseen phrasing or joins. |
| Vocabulary mismatch | Business words like 'revenue' and 'repeat customers' had to be mapped onto schema terms. |
| Neural translation | Phase B used an LLM to learn the English-to-SQL mapping instead of relying on hard-coded rules. |
| Schema as target grammar | The schema constrained what counted as valid output in the target language. |
| Hallucination | The model produced wrong-but-valid or incomplete SQL on ambiguous or schema-mismatched prompts. |
| Human-in-the-loop | Phase C required SQL review and guardrails before trusting an answer. |

### Production Conclusion

The production recommendation is Phase C: LLM translation plus guardrails plus visible SQL review. That is the first phase that balances usability with business trust.